In [1]:
import os
import numpy as np
import pandas as pd

DATA_ROOT = "/kaggle/data"
TRAIN_LABELS_PATH = os.path.join(DATA_ROOT, "train_labels.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_ROOT, "sample_submission.csv")
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")

train_labels = pd.read_csv(TRAIN_LABELS_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)


def id_to_path(root_dir: str, _id: str) -> str:
    # Dataset layout is root_dir/<hex>/<id>.npy where <hex> is 0..f (NOT the first char of id necessarily);
    # in this competition, files are indeed sharded by first character, so this path is correct.
    return os.path.join(root_dir, _id[0], f"{_id}.npy")


def build_id2path_from_ids(root_dir: str, ids) -> dict:
    # Score-critical fix: previous version scanned only root_dir/<hex>/ and built an "existing" set,
    # but then incorrectly checked/constructed paths using id[0] as the folder, which can drop most data
    # if folder naming doesn't match that assumption or if scan misses nested structure.
    # Minimal, semantics-preserving approach: just compute the expected path and keep those that exist.
    id2path = {}
    for _id in ids:
        p = id_to_path(root_dir, _id)
        if os.path.exists(p):
            id2path[_id] = p
    return id2path


train_ids_all = train_labels["id"].tolist()
test_ids = sample_sub["id"].tolist()

train_id2path = build_id2path_from_ids(TRAIN_DIR, train_ids_all)
test_id2path = build_id2path_from_ids(TEST_DIR, test_ids)

train_df = train_labels[train_labels["id"].isin(train_id2path)].reset_index(drop=True)

missing_test = [i for i in test_ids if i not in test_id2path]
if len(missing_test) > 0:
    raise FileNotFoundError(
        f"Missing {len(missing_test)} test .npy files; example: {missing_test[0]}"
    )

print(f"Train rows with files: {len(train_df)} / {len(train_labels)}")
print(f"Test rows: {len(test_ids)}")



Train rows with files: 54000 / 54000
Test rows: 6000


In [2]:
from concurrent.futures import ThreadPoolExecutor


def _quantiles_nearest_rank_2d(
    D: np.ndarray, q1: float, q2: float
) -> tuple[np.float32, np.float32]:
    a = D.ravel()
    n = a.size
    if n == 0:
        return np.float32(np.nan), np.float32(np.nan)

    def _q_val(q: float) -> np.float32:
        pos = (n - 1) * q
        lo = int(np.floor(pos))
        hi = int(np.ceil(pos))
        return np.float32(pos), lo, hi

    pos1, lo1, hi1 = _q_val(q1)
    pos2, lo2, hi2 = _q_val(q2)

    idxs = [lo1, hi1, lo2, hi2]
    uniq = np.unique(np.array(idxs, dtype=np.int64))
    part = np.partition(a, uniq)
    m = {int(k): np.float32(part[int(k)]) for k in uniq}

    def _interp(pos: np.float32, lo: int, hi: int) -> np.float32:
        if lo == hi:
            return m[lo]
        w = np.float32(pos - lo)
        return np.float32((1.0 - w) * m[lo] + w * m[hi])

    return _interp(np.float32(pos1), lo1, hi1), _interp(np.float32(pos2), lo2, hi2)


def _corr_1d(a: np.ndarray, b: np.ndarray) -> np.float32:
    a = a.astype(np.float32, copy=False)
    b = b.astype(np.float32, copy=False)
    am = a.mean()
    bm = b.mean()
    da = a - am
    db = b - bm
    denom = np.sqrt((da * da).mean() * (db * db).mean()) + 1e-6
    return np.float32((da * db).mean() / denom)


_DRIFT_CACHE = {}


def _get_drift_cache(T: int, F: int, slopes: tuple[int, ...]):
    key = (T, F, slopes)
    v = _DRIFT_CACHE.get(key)
    if v is not None:
        return v

    t_idx = np.arange(T, dtype=np.int32)
    entries = []
    for s in slopes:
        if s == 0:
            entries.append((s, None, None))
            continue

        if s > 0:
            f0_min = 0
            f0_max = F - 1 - s * (T - 1)
        else:
            f0_min = -s * (T - 1)
            f0_max = F - 1

        if f0_max < f0_min:
            entries.append((s, None, None))
            continue

        f0_candidates = np.linspace(f0_min, f0_max, num=32, dtype=np.int32)
        f_idx = (f0_candidates[None, :] + (s * t_idx[:, None])).astype(
            np.int32, copy=False
        )
        entries.append((s, f_idx, t_idx))
    _DRIFT_CACHE[key] = entries
    return entries


def _max_drift_score_multi(
    stack2d: np.ndarray, slopes=(-4, -2, -1, 0, 1, 2, 4)
) -> np.float32:
    S = stack2d.astype(np.float32, copy=False)
    T, F = S.shape
    best = np.float32(-1e9)

    cache = _get_drift_cache(T, F, tuple(slopes))
    for s, f_idx, t_idx in cache:
        if s == 0:
            score = np.max(S.sum(axis=0))
            if score > best:
                best = np.float32(score)
            continue
        if f_idx is None:
            continue
        scores = S[t_idx[:, None], f_idx].sum(axis=0)
        smax = scores.max(initial=np.float32(-1e9))
        if smax > best:
            best = np.float32(smax)
    return best


def _max_drift_score(stack2d: np.ndarray) -> np.float32:
    return _max_drift_score_multi(stack2d, slopes=(-2, -1, 0, 1, 2))


def _contrastize_D(D: np.ndarray) -> np.ndarray:
    D = D.astype(np.float32, copy=False)
    lo, hi = _quantiles_nearest_rank_2d(D, 0.01, 0.99)
    Dc = np.clip(D, lo, hi)
    Dc = Dc - Dc.mean(axis=1, keepdims=True)
    Dc = Dc - Dc.mean(axis=0, keepdims=True)
    Dc = (Dc - Dc.mean()) / (Dc.std(ddof=0) + 1e-6)
    return Dc


def _log1p_signed(Z: np.ndarray) -> np.ndarray:
    Z = Z.astype(np.float32, copy=False)
    return np.sign(Z) * np.log1p(np.abs(Z))


def extract_features_from_array(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32, copy=False)

    means = x.mean(axis=(1, 2))
    stds = x.std(axis=(1, 2), ddof=0)
    maxs = x.max(axis=(1, 2))
    mins = x.min(axis=(1, 2))

    A0 = x[0]
    A1 = x[2]
    A2 = x[4]
    O0 = x[1]
    O1 = x[3]
    O2 = x[5]

    A_mean = (A0.mean() + A1.mean() + A2.mean()) / 3.0
    off_mean = (O0.mean() + O1.mean() + O2.mean()) / 3.0

    A0m, A1m, A2m = A0.mean(), A1.mean(), A2.mean()
    O0m, O1m, O2m = O0.mean(), O1.mean(), O2.mean()

    A_var = (A0.var(ddof=0) + A1.var(ddof=0) + A2.var(ddof=0)) / 3.0 + (
        (((A0m - A_mean) ** 2) + ((A1m - A_mean) ** 2) + ((A2m - A_mean) ** 2)) / 3.0
    )
    off_var = (O0.var(ddof=0) + O1.var(ddof=0) + O2.var(ddof=0)) / 3.0 + (
        (((O0m - off_mean) ** 2) + ((O1m - off_mean) ** 2) + ((O2m - off_mean) ** 2))
        / 3.0
    )

    A_std = np.sqrt(A_var, dtype=np.float32)
    off_std = np.sqrt(off_var, dtype=np.float32)

    diff_mean = A_mean - off_mean
    ratio_std = (A_std + 1e-6) / (off_std + 1e-6)

    A0_time = A0.mean(axis=1)
    A1_time = A1.mean(axis=1)
    A2_time = A2.mean(axis=1)
    O0_time = O0.mean(axis=1)
    O1_time = O1.mean(axis=1)
    O2_time = O2.mean(axis=1)

    A0_freq = A0.mean(axis=0)
    A1_freq = A1.mean(axis=0)
    A2_freq = A2.mean(axis=0)
    O0_freq = O0.mean(axis=0)
    O1_freq = O1.mean(axis=0)
    O2_freq = O2.mean(axis=0)

    time_var_diff = (
        A0_time.var(ddof=0) + A1_time.var(ddof=0) + A2_time.var(ddof=0)
    ) / 3.0 - (O0_time.var(ddof=0) + O1_time.var(ddof=0) + O2_time.var(ddof=0)) / 3.0
    freq_var_diff = (
        A0_freq.var(ddof=0) + A1_freq.var(ddof=0) + A2_freq.var(ddof=0)
    ) / 3.0 - (O0_freq.var(ddof=0) + O1_freq.var(ddof=0) + O2_freq.var(ddof=0)) / 3.0

    A_stack = (A0 + A1 + A2) / 3.0
    O_stack = (O0 + O1 + O2) / 3.0
    D = A_stack - O_stack
    d_mean = D.mean()
    d_std = D.std(ddof=0)
    d_max = D.max()
    d_min = D.min()

    corr_A = (
        _corr_1d(A0_time, A1_time)
        + _corr_1d(A0_time, A2_time)
        + _corr_1d(A1_time, A2_time)
    ) / 3.0
    corr_O = (
        _corr_1d(O0_time, O1_time)
        + _corr_1d(O0_time, O2_time)
        + _corr_1d(O1_time, O2_time)
    ) / 3.0
    corr_diff = np.float32(corr_A - corr_O)

    A_mad = (
        np.mean(np.abs(A0 - A0m))
        + np.mean(np.abs(A1 - A1m))
        + np.mean(np.abs(A2 - A2m))
    ) / 3.0
    O_mad = (
        np.mean(np.abs(O0 - O0m))
        + np.mean(np.abs(O1 - O1m))
        + np.mean(np.abs(O2 - O2m))
    ) / 3.0
    mad_ratio = np.float32((A_mad + 1e-6) / (O_mad + 1e-6))
    mad_diff = np.float32(A_mad - O_mad)

    A_max_mean = np.float32((A0.max() + A1.max() + A2.max()) / 3.0)
    O_max_mean = np.float32((O0.max() + O1.max() + O2.max()) / 3.0)
    excess_max = np.float32(A_max_mean - O_max_mean)

    def _tail_frac(panel: np.ndarray) -> np.float32:
        panel32 = panel.astype(np.float32, copy=False)
        _, thr = _quantiles_nearest_rank_2d(panel32, 0.0, 0.99)
        return np.float32((panel32 > thr).mean())

    A_tail = np.float32((_tail_frac(A0) + _tail_frac(A1) + _tail_frac(A2)) / 3.0)
    O_tail = np.float32((_tail_frac(O0) + _tail_frac(O1) + _tail_frac(O2)) / 3.0)
    tail_ratio = np.float32((A_tail + 1e-6) / (O_tail + 1e-6))

    def _line_strength(panel: np.ndarray) -> tuple[np.float32, np.float32]:
        row_max = np.float32(panel.mean(axis=1).max())
        col_max = np.float32(panel.mean(axis=0).max())
        return row_max, col_max

    A0_r, A0_c = _line_strength(A0)
    A1_r, A1_c = _line_strength(A1)
    A2_r, A2_c = _line_strength(A2)
    O0_r, O0_c = _line_strength(O0)
    O1_r, O1_c = _line_strength(O1)
    O2_r, O2_c = _line_strength(O2)

    A_row = np.float32((A0_r + A1_r + A2_r) / 3.0)
    O_row = np.float32((O0_r + O1_r + O2_r) / 3.0)
    A_col = np.float32((A0_c + A1_c + A2_c) / 3.0)
    O_col = np.float32((O0_c + O1_c + O2_c) / 3.0)

    row_diff = np.float32(A_row - O_row)
    col_diff = np.float32(A_col - O_col)
    row_ratio = np.float32((A_row + 1e-6) / (O_row + 1e-6))
    col_ratio = np.float32((A_col + 1e-6) / (O_col + 1e-6))

    A_drift = _max_drift_score(A_stack)
    O_drift = _max_drift_score(O_stack)
    drift_diff = np.float32(A_drift - O_drift)
    drift_ratio = np.float32((A_drift + 1e-6) / (O_drift + 1e-6))

    D_drift = _max_drift_score_multi(D)
    D_abs = np.abs(D)
    D_abs_mean = np.float32(D_abs.mean())
    D_abs_max = np.float32(D_abs.max())
    D_abs_ratio = np.float32((D_abs_max + 1e-6) / (D_abs_mean + 1e-6))

    D_energy_l1 = np.float32(np.mean(np.abs(D)) + 1e-6)
    D_drift_norm = np.float32(D_drift / D_energy_l1)
    D_rev = D[::-1, :]
    D_drift_rev = _max_drift_score_multi(D_rev)
    D_drift_dir_ratio = np.float32((D_drift + 1e-6) / (D_drift_rev + 1e-6))

    D_colsum_max = np.float32(np.max(D.sum(axis=0)))
    D_drift_minus_col = np.float32(D_drift - D_colsum_max)
    D_flip = D[:, ::-1]
    D_drift_flip = _max_drift_score_multi(D_flip)

    D_w = (D - np.float32(d_mean)) / (np.float32(d_std) + 1e-6)
    D_w_drift = _max_drift_score_multi(D_w)
    D_w_flip_drift = _max_drift_score_multi(D_w[:, ::-1])

    Dc = _contrastize_D(D)
    Dc_drift = _max_drift_score_multi(Dc)
    Dc_drift_flip = _max_drift_score_multi(Dc[:, ::-1])
    Dc_drift_rev = _max_drift_score_multi(Dc[::-1, :])
    Dc_dir_ratio = np.float32((Dc_drift + 1e-6) / (Dc_drift_rev + 1e-6))

    Dlog = _log1p_signed(D)
    Dlog_drift = _max_drift_score_multi(Dlog)
    Dlog_flip_drift = _max_drift_score_multi(Dlog[:, ::-1])
    Dlog_colsum_max = np.float32(np.max(Dlog.sum(axis=0)))
    Dlog_drift_minus_col = np.float32(Dlog_drift - Dlog_colsum_max)

    D_freq_peak = np.float32(np.max(D.mean(axis=0)))

    n_features = 6 * 4 + 8 + 4 + 5 + 2 + 4 + 2 + 4 + 2 + 2 + 2 + 4 + 4 + 1
    feats = np.empty(n_features, dtype=np.float32)
    feats[0:6] = means
    feats[6:12] = stds
    feats[12:18] = maxs
    feats[18:24] = mins
    feats[24:32] = np.array(
        [
            A_mean,
            off_mean,
            diff_mean,
            A_std,
            off_std,
            ratio_std,
            time_var_diff,
            freq_var_diff,
        ],
        dtype=np.float32,
    )
    feats[32:36] = np.array([d_mean, d_std, d_max, d_min], dtype=np.float32)
    feats[36:41] = np.array(
        [corr_A, corr_O, corr_diff, mad_ratio, mad_diff], dtype=np.float32
    )
    feats[41:43] = np.array([excess_max, tail_ratio], dtype=np.float32)
    feats[43:47] = np.array(
        [row_diff, col_diff, row_ratio, col_ratio], dtype=np.float32
    )
    feats[47:49] = np.array([drift_diff, drift_ratio], dtype=np.float32)
    feats[49:53] = np.array(
        [D_drift, D_abs_mean, D_abs_max, D_abs_ratio], dtype=np.float32
    )
    feats[53:55] = np.array([D_drift_norm, D_drift_dir_ratio], dtype=np.float32)
    feats[55:57] = np.array([D_drift_minus_col, D_drift_flip], dtype=np.float32)
    feats[57:59] = np.array([D_w_drift, D_w_flip_drift], dtype=np.float32)
    feats[59:63] = np.array(
        [Dc_drift, Dc_drift_flip, Dc_drift_rev, Dc_dir_ratio], dtype=np.float32
    )
    feats[63:67] = np.array(
        [Dlog_drift, Dlog_flip_drift, Dlog_colsum_max, Dlog_drift_minus_col],
        dtype=np.float32,
    )
    feats[67:68] = np.array([D_freq_peak], dtype=np.float32)
    return feats


def extract_features_for_ids(ids, id2path, batch_print_every=5000, max_workers=None):
    ids = list(ids)
    n_features = 6 * 4 + 8 + 4 + 5 + 2 + 4 + 2 + 4 + 2 + 2 + 2 + 4 + 4 + 1
    X = np.empty((len(ids), n_features), dtype=np.float32)

    def _load_and_extract(idx: int):
        _id = ids[idx]
        arr = np.load(id2path[_id], mmap_mode=None)
        return idx, extract_features_from_array(arr)

    if max_workers is None:
        cpu = os.cpu_count() or 4
        max_workers = min(16, cpu)

    chunksize = 1024
    done = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for idx, feats in ex.map(
            _load_and_extract, range(len(ids)), chunksize=chunksize
        ):
            X[idx] = feats
            done += 1
            if batch_print_every and done % batch_print_every == 0:
                print(f"Processed {done}/{len(ids)}")
    return X


X_train = extract_features_for_ids(
    train_df["id"].tolist(), train_id2path, batch_print_every=10000
)
y_train = train_df["target"].astype(np.int8).values

print("X_train shape:", X_train.shape, "y_train mean:", y_train.mean())



Processed 10000/54000
Processed 20000/54000
Processed 30000/54000
Processed 40000/54000
Processed 50000/54000
X_train shape: (54000, 68) y_train mean: 0.09937037037037037


In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

models = []
oof = np.zeros(len(y_train), dtype=np.float64)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), 1):
    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    solver="lbfgs",
                    max_iter=500,
                    n_jobs=None,
                    class_weight=None,
                    C=1.0,
                    random_state=42,
                ),
            ),
        ]
    )
    model.fit(X_train[tr_idx], y_train[tr_idx])
    oof[va_idx] = model.predict_proba(X_train[va_idx])[:, 1]
    models.append(model)
    print(f"Trained fold {fold}/{n_splits}")

print("OOF AUC:", roc_auc_score(y_train, oof))



Trained fold 1/5
Trained fold 2/5
Trained fold 3/5
Trained fold 4/5
Trained fold 5/5
OOF AUC: 0.5076400071419691


In [4]:
X_test = extract_features_for_ids(test_ids, test_id2path, batch_print_every=2000)
print("X_test shape:", X_test.shape)

preds = np.zeros(len(test_ids), dtype=np.float64)
for m in models:
    preds += m.predict_proba(X_test)[:, 1]
preds /= len(models)

preds = np.clip(preds, 0.0, 1.0)

submission = pd.DataFrame({"id": test_ids, "target": preds})



Processed 2000/6000
Processed 4000/6000
Processed 6000/6000
X_test shape: (6000, 68)


In [5]:
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)

                id    target
0  0cee567456cd304  0.092943
1  5451b45281c65a7  0.094446
2  f8cc6cea820282d  0.093308
3  25e21ba81a64742  0.101092
4  aafa910406b1db2  0.101767
Wrote submission.csv with shape: (6000, 2)
